# ML4BI2 — Exam Part 2
## Freiburg Groceries — Unified Notebook

**Author:** Tomáš Kováč
**Goal:** one notebook that (a) trains a *from-scratch* CNN and a *pretrained + fine-tuned* CNN on the **full train + validation** set, saves both best checkpoints to Google Drive, and (b) on exam day loads them and evaluates on `test/`.


## How to use this notebook

The notebook is split into two sections:

**Section A — Training before the exam.**
Runs end-to-end:
1. Data exploration + duplicate removal between `train/` and `validation/`.
2. From-scratch climb (Models 1 → 2 → 3 → 4) on the standard `train/val` split — this is the *educational story* I want to show the teacher: how each ingredient (augmentation, residuals, BN, dropout) improves validation accuracy.
3. **Final retraining** of the winning architecture on `train + val` combined (with a 10% holdout for monitoring) → saves `best_from_scratch_Freiburg.keras` to Drive.
4. **Pretrained EfficientNetV2B0** — phase 1 (frozen base) + phase 2 (fine-tune top blocks), also on `train + val` combined → saves `best_finetuned_Freiburg.keras` to Drive.

**Section B — Exam day (no training).**
Loads the two best checkpoints from Drive, builds a test `tf.data` pipeline from `test/`, and reports per-class accuracy + confusion matrix.

> **Tip for the reader / examiner:** if you want to skip the educational climb and only evaluate the final models, jump straight to *Section B*. Both best models are already on my Drive.


---
# 0. Setup — mount Drive, configure paths, switch on GPU

Colab: *Runtime → Change runtime type → GPU* (T4 is enough).

All four cells of this setup are needed for **both** Section A (training) and Section B (exam-day inference).


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


Mounted at /content/gdrive


In [ ]:
import zipfile, os

# --- Paths ------------------------------------------------------------------
# The dataset zip lives on Drive. We extract it to /content (local SSD) so that
# data-loading is fast — every epoch reads each image repeatedly, and Drive I/O
# would dominate training time otherwise.
ZIP_PATH    = '/content/gdrive/MyDrive/ML2/exam_ML4BI2_2026_Freiburg.zip'
EXTRACT_DIR = '/content/data'                       # local SSD — fast
DRIVE_DIR   = '/content/gdrive/MyDrive/ML2'         # persistent — where we save models
MODEL_DIR   = DRIVE_DIR

if not os.path.exists(os.path.join(EXTRACT_DIR, 'exam_ML4BI2_2026', 'train')):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print(f'Extracted to {EXTRACT_DIR}')
else:
    print('Already extracted to local SSD.')

# --- Folder structure -------------------------------------------------------
# The dataset has three splits:
#   train/      ~158 images per class (25 classes)
#   validation/ ~20 images per class  (after we remove duplicates of train/)
#   test/       provided at the exam, same 25-folder structure
TRAIN_DIR = os.path.join(EXTRACT_DIR, 'exam_ML4BI2_2026', 'train')
VAL_DIR   = os.path.join(EXTRACT_DIR, 'exam_ML4BI2_2026', 'validation')
TEST_DIR  = os.path.join(EXTRACT_DIR, 'exam_ML4BI2_2026', 'test')   # used in Section B

# --- Saved-model file names -------------------------------------------------
# Educational climb (Section A.2): we train four models on the train/val split
# only so we can compare them. These checkpoints are throw-away.
MODEL_M1 = os.path.join(MODEL_DIR, 'climb_M1_simple.keras')
MODEL_M2 = os.path.join(MODEL_DIR, 'climb_M2_aug_bn.keras')
MODEL_M3 = os.path.join(MODEL_DIR, 'climb_M3_residual.keras')
MODEL_M4 = os.path.join(MODEL_DIR, 'climb_M4_playmodel.keras')

# Final deliverables (Section A.4 + A.5): trained on train + val combined.
# These are the two files Section B loads from Drive.
BEST_FROM_SCRATCH = os.path.join(MODEL_DIR, 'best_from_scratch_Freiburg.keras')
BEST_FINETUNED    = os.path.join(MODEL_DIR, 'best_finetuned_Freiburg.keras')

print('TRAIN_DIR        :', TRAIN_DIR)
print('VAL_DIR          :', VAL_DIR)
print('TEST_DIR         :', TEST_DIR, '(used only on exam day)')
print('MODEL_DIR        :', MODEL_DIR)
print('BEST_FROM_SCRATCH:', BEST_FROM_SCRATCH)
print('BEST_FINETUNED   :', BEST_FINETUNED)


FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/ML2/exam_ML4BI2_2026_Freiburg.zip'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TF version:', tf.__version__)

# Reproducibility — same seed pattern as the in-class exercise E5. Not 100%
# deterministic on GPU (cuDNN ops), but it greatly reduces run-to-run variance.
keras.utils.set_random_seed(42)


---
# **SECTION A — Training**

This section runs the whole training pipeline. On a Colab T4 the full Section A takes roughly **2.5–3 hours**. It writes two files to `MODEL_DIR`:

| File | Trained on | Architecture |
|---|---|---|
| `best_from_scratch_Freiburg.keras` | train + val combined | Model 4 (residual CNN built from scratch) |
| `best_finetuned_Freiburg.keras`    | train + val combined | EfficientNetV2B0, top blocks fine-tuned |

Both files are loaded in Section B without retraining.


## 1. Universal workflow of machine learning


### 1.1 Defining the task

- **Type:** supervised, multi-class, single-label image classification.
- **Input:** a 224×224 RGB photograph of a grocery product.
- **Output:** one of **25** product-category labels.
- **Hypothesis:** the visual appearance of a grocery item is predictive of its category — a CNN is the natural choice.

### 1.2 Key metrics

Random guessing on 25 balanced classes ≈ 1/25 = **4 %** accuracy. I monitor:
- **Categorical accuracy** — the primary metric (also what the exam is graded on).
- **Top-3 accuracy** — a useful diagnostic: if top-3 is high but top-1 is low, the model has signal but confuses look-alike packages.


### 1.3 Understand the data

Before building any model, a quick EDA to make sure the assumptions in 1.1 actually hold.


In [ ]:
# Class names are the sub-folder names under train/. Sorted alphabetically so
# that the index → name mapping is deterministic between runs.
class_names = sorted([c for c in os.listdir(TRAIN_DIR) if not c.startswith('.')])
num_classes = len(class_names)
print('Number of classes:', num_classes)
print('Classes:', class_names)


In [ ]:
# How many images per class? Imbalance would force us to weight the loss.
train_counts = {c: len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in class_names}
val_counts   = {c: len(os.listdir(os.path.join(VAL_DIR,   c))) for c in class_names}

fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].bar(train_counts.keys(), train_counts.values())
ax[0].set_title('Train images / class')
ax[1].bar(val_counts.keys(),   val_counts.values(), color='orange')
ax[1].set_title('Validation images / class')
for a in ax:
    a.tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.show()

print(f'Train total = {sum(train_counts.values())}  |  Val total = {sum(val_counts.values())}')
print(f'Train per class: mean {np.mean(list(train_counts.values())):.1f}, '
      f'min {min(train_counts.values())}, max {max(train_counts.values())}')


In [ ]:
# Visual sanity check — one image per class. Confirms that the folder labels
# match the actual content and that 224x224 is a reasonable input size.
fig, axes = plt.subplots(5, 5, figsize=(15, 15))
for ax, cls in zip(axes.ravel(), class_names):
    fname = os.listdir(os.path.join(TRAIN_DIR, cls))[0]
    img = keras.utils.load_img(os.path.join(TRAIN_DIR, cls, fname))
    ax.imshow(img); ax.set_title(cls, fontsize=10); ax.axis('off')
plt.tight_layout(); plt.show()


**Observations:**

- ~158 images per class on average — a **small** dataset. Without data augmentation and dropout we will overfit immediately.
- The classes are roughly balanced, so we do not need class weighting in the loss.
- Random guessing accuracy = 1/25 = 4 %, so anything noticeably above 4 % already shows that the model is learning something.
- All images are 224×224 RGB, matching common ImageNet-pretrained input sizes — that will pay off in Section A.5 when we plug in EfficientNetV2B0.


### 1.4 Data-leakage check — remove duplicate images across splits

Some images in `validation/` are **byte-identical** copies of files in `train/`. If we leave them in, the model trivially recognises them and our val accuracy is inflated — exactly the *information leak* Chollet warns about in §5.2.

I MD5-hash every image and delete any `validation/` file whose hash also appears in `train/`. To make it fast, I first group files by size — files with a unique size can't possibly be duplicates, so I only hash the size-collisions.


In [ ]:
import hashlib
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

def list_images(root):
    """Return all image paths under a directory (skipping hidden files)."""
    out = []
    if not os.path.exists(root):
        return out
    for cls in sorted(os.listdir(root)):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            continue
        for f in sorted(os.listdir(cls_dir)):
            if f.startswith('.'):
                continue
            out.append(os.path.join(cls_dir, f))
    return out

def md5_of(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

train_paths = list_images(TRAIN_DIR)
val_paths   = list_images(VAL_DIR)
print(f'Before dedup:  train={len(train_paths)}  |  val={len(val_paths)}')

# Optimisation: hash only files whose size collides with another file.
size_groups = defaultdict(list)
for p in train_paths + val_paths:
    size_groups[os.path.getsize(p)].append(p)
candidates = [p for ps in size_groups.values() if len(ps) > 1 for p in ps]

# Hash in parallel (I/O-bound → threads are fine).
with ThreadPoolExecutor(max_workers=16) as pool:
    hashes = dict(zip(candidates, pool.map(md5_of, candidates)))

train_hash_set = {hashes[p] for p in train_paths if p in hashes}
duplicates_in_val = [p for p in val_paths if p in hashes and hashes[p] in train_hash_set]

print(f'\nFound {len(duplicates_in_val)} val images that are byte-identical to a train image.')
for p in duplicates_in_val:
    rel = os.path.relpath(p, VAL_DIR)
    print(f'  removing val/{rel}')
    os.remove(p)

val_paths_after = list_images(VAL_DIR)
print(f'\nAfter dedup:   train={len(train_paths)}  |  val={len(val_paths_after)}')


### 1.5 Prepare the data — `tf.data` pipelines

I use `keras.utils.image_dataset_from_directory` (the same helper used in the CH 8 walk-through) to build a `tf.data.Dataset` that streams batches lazily — no big NumPy arrays sitting in RAM.

**Design decisions:**
- `label_mode='categorical'` → one-hot labels paired with `categorical_crossentropy`.
- `Rescaling(1./255)` is applied **inside** the model, not in the dataset. That way the preprocessing travels with the saved `.keras` file → on exam day, `model.evaluate(test_dataset)` is enough — no separate preprocessing step to remember.
- `shuffle=True` only on train (val needs deterministic order for reproducible evaluation).
- `BATCH_SIZE=32` — a standard size that fits comfortably on a T4 GPU.
- `prefetch(AUTOTUNE)` overlaps CPU preprocessing with GPU forward/backward.


In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 64

train_dataset = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=42,
)

val_dataset = keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
)

train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
val_dataset   = val_dataset.prefetch(tf.data.AUTOTUNE)

# Sanity check on the first batch — confirms shape and pixel range.
for x, y in train_dataset.take(1):
    print('Batch shape :', x.shape, '  label shape:', y.shape)
    print('Pixel range :', float(tf.reduce_min(x)), '→', float(tf.reduce_max(x)))


**Confirmed from the sanity check:**
1. Pixels live in `[0, 255]` → `Rescaling(1./255)` is the correct preprocessing layer (or `efficientnet_v2.preprocess_input` for the pretrained model in A.5).
2. Input shape `(224, 224, 3)` → matches the Keras `Input` layer of every model below.
3. Labels shape `(batch, 25)` → final `Dense(25, activation='softmax')`.


## 2. Convnets from scratch — Chollet's three-step climb

Chollet's advice in CH 8/9 is: *don't try to build the perfect model on the first try*. Climb the ladder in steps so you understand what each ingredient buys you.

| # | Model | New ingredients | Hypothesis I want to verify |
|---|---|---|---|
| 1 | **Simple convnet (baseline)** | none — plain Conv → MaxPool stack | Has *signal* but overfits hard. |
| 2 | **+ Augmentation + BN + SeparableConv** | random transforms, batch norm, depth-wise separable convs, GAP head, dropout | Closes the train/val gap; val accuracy roughly doubles. |
| 3 | **+ Residual connections** | shortcut additions (CH 9) | Allows depth to actually help; another jump in val accuracy. |
| 4 | **Improved residual (final)** | proper Conv-BN-ReLU order, downsample blocks, `SpatialDropout2D` | The winner — used as the final from-scratch deliverable. |

Each climb step is saved with `ModelCheckpoint(save_best_only=True, monitor='val_loss')`. After step 4 I retrain the winning architecture on **train + val combined** (Section A.4) — that file is the actual deliverable.


### Reusable plotting helpers

Two helpers used after every training run. The first plots train vs. validation curves; the second prints the best epoch (lowest `val_loss`), which is the epoch that `ModelCheckpoint` saved.


In [ ]:
def plot_history(history, title=''):
    """Side-by-side loss and accuracy curves for train vs validation."""
    h = history.history
    ep = range(1, len(h['loss']) + 1)
    plt.figure(figsize=(13, 4))
    plt.subplot(1, 2, 1)
    plt.plot(ep, h['loss'],     'bo-', label='Training')
    if 'val_loss' in h:
        plt.plot(ep, h['val_loss'], 'ro-', label='Validation')
    plt.title(f'{title} — loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(ep, h['accuracy'],     'bo-', label='Training')
    if 'val_accuracy' in h:
        plt.plot(ep, h['val_accuracy'], 'ro-', label='Validation')
    plt.title(f'{title} — accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
    plt.tight_layout(); plt.show()

def print_best_val(history):
    """Report the epoch with lowest val_loss (= what ModelCheckpoint saved)."""
    if 'val_loss' not in history.history:
        i = int(np.argmin(history.history['loss']))
        print(f'(No val set) Best epoch by train loss: {i+1}')
        print(f'  loss          = {history.history["loss"][i]:.4f}')
        print(f'  accuracy      = {history.history["accuracy"][i]:.4f}')
        return
    i = int(np.argmin(history.history['val_loss']))
    print(f'Best epoch (lowest val_loss): {i+1}')
    print(f'  val_loss          = {history.history["val_loss"][i]:.4f}')
    print(f'  val_accuracy      = {history.history["val_accuracy"][i]:.4f}')
    if 'val_top3_accuracy' in history.history:
        print(f'  val_top3_accuracy = {history.history["val_top3_accuracy"][i]:.4f}')


### Shared data-augmentation block

Defined here once and re-used by Models 2, 3, 4 and the final retrained model.

> **Why augmentation lives *inside* the model (not in the dataset pipeline):**
> `RandomFlip`, `RandomRotation`, etc. are no-ops at inference (`training=False`). Putting them inside the model means the saved `.keras` file behaves correctly during `model.evaluate(test_dataset)` — no need to remember separate train- vs. eval-time pipelines.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
], name='data_augmentation')

# Visual sanity check — 8 random augmentations of one training image
for x, _ in train_dataset.take(1):
    plt.figure(figsize=(10, 4))
    for i in range(8):
        aug = data_augmentation(x)
        plt.subplot(2, 4, i + 1)
        plt.imshow(aug[0].numpy().astype('uint8')); plt.axis('off')
    plt.suptitle('8 random augmentations of one training image')
    plt.tight_layout(); plt.show()
    break


### 2.1 Model 1 — simple convnet (the baseline)

**Goal of this step:** establish a baseline and *intentionally* observe overfitting. This is the textbook CH 8 architecture (the dogs-vs-cats convnet) adapted for 224×224 inputs and 25 outputs.

**What's missing — on purpose:**
- No augmentation → the model sees each image only as it is.
- No dropout, no BN, no L2 → no regularisation.
- A big `Flatten()` right before the classifier head → ~50k features × 25 classes = ~1.2 M parameters in the head alone, which is the main overfit driver.

**Expectation:** training accuracy climbs fast while validation accuracy stagnates — a textbook overfit curve. That gap is the whole point of this step: it motivates everything we add next.


In [ ]:
# --- Architecture: 5 Conv blocks → Flatten → Dense(25, softmax) ------------
inputs = keras.Input(shape=(224, 224, 3))
x = layers.Rescaling(1./255)(inputs)                  # scale pixels to [0, 1]

# Progressively more filters as spatial size shrinks — the classic "feature
# pyramid": shallow layers detect edges/colors, deep layers detect parts/objects.
x = layers.Conv2D(32,  3, activation='relu')(x); x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(64,  3, activation='relu')(x); x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(128, 3, activation='relu')(x); x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(256, 3, activation='relu')(x); x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(256, 3, activation='relu')(x)
x = layers.Flatten()(x)                               # ← parameter-heavy head
outputs = layers.Dense(num_classes, activation='softmax')(x)

model_m1 = keras.Model(inputs, outputs, name='M1_simple')
model_m1.summary()


In [ ]:
model_m1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')],
)

# ModelCheckpoint saves the epoch with the lowest val_loss to Drive. Even if
# later epochs overfit harder, the saved file is the "best version".
callbacks_m1 = [
    keras.callbacks.ModelCheckpoint(MODEL_M1, save_best_only=True, monitor='val_loss'),
]

history_m1 = model_m1.fit(
    train_dataset,
    epochs=30,
    validation_data=val_dataset,
    callbacks=callbacks_m1,
)


In [ ]:
plot_history(history_m1, 'M1 — simple convnet')
print_best_val(history_m1)


**Interpretation of Model 1.** Training accuracy rockets up while validation accuracy stalls — the train/val gap is wide. The model has *signal* (val_acc ≫ 4 % random), but it has memorised the training set rather than learning generalisable features. This is the overfit Chollet describes in CH 5/6, and it justifies everything we add in Model 2.


### 2.2 Model 2 — `+ augmentation + BN + SeparableConv + GAP + dropout`

**Goal of this step:** close the train/val gap by (a) injecting variability into the training data and (b) using standard regularisers.

**What I add and why:**

- **Data augmentation** (random flip / rotation / zoom / translation / contrast). Each epoch the model sees slightly perturbed versions of every image — effectively a larger dataset. From CH 8.
- **Batch normalization** before each activation. Stabilises gradients and lets us train slightly deeper. From CH 9.
- **Separable convolutions** (`SeparableConv2D`) for blocks 2–4. Same expressive power as `Conv2D` but ~9× fewer parameters → less to overfit. From CH 9.
  - The **first block stays a plain `Conv2D`** because separable convs assume the input channels are already decorrelated — not the case for raw RGB.
- **Global Average Pooling** instead of `Flatten()` → no giant dense layer at the head → much less overfitting at the classifier. From CH 9.
- **Dropout(0.5)** before the final softmax.


In [ ]:
# --- Architecture: stem Conv2D → 4 SeparableConv blocks → GAP → Dense -----
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

# Stem: plain Conv2D — RGB channels are not yet decorrelated.
x = layers.Conv2D(32, 5, use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

# 4 SeparableConv blocks (CH 9 style).
for size in [64, 128, 256, 512]:
    x = layers.SeparableConv2D(size, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)

# Lightweight head: GAP → Dropout → Dense(25, softmax).
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model_m2 = keras.Model(inputs, outputs, name='M2_aug_bn')
model_m2.summary()


In [ ]:
model_m2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')],
)

callbacks_m2 = [
    keras.callbacks.ModelCheckpoint(MODEL_M2, save_best_only=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
]

history_m2 = model_m2.fit(
    train_dataset,
    epochs=60,
    validation_data=val_dataset,
    callbacks=callbacks_m2,
)


In [ ]:
plot_history(history_m2, 'M2 — + augmentation + BN + SeparableConv')
print_best_val(history_m2)


**Interpretation of Model 2.** The train/val gap is dramatically smaller than in Model 1 — augmentation forces the model to generalise instead of memorise, and BN+GAP slow down the dense-head overfit. Validation accuracy roughly **doubles** compared to Model 1. This is exactly the payoff Chollet promises in CH 8.

But the architecture is still shallow, and val accuracy is still well below where a careful from-scratch model can go on this dataset. Time to add depth — via **residual connections**.


### 2.3 Model 3 — residual + augmentation (CH 9)

**Goal of this step:** make depth actually help. Plain stacks of convolutions stop improving (or even start degrading) once they get too deep — gradients vanish. **Residual connections** (the `x + F(x)` shortcut from ResNet) fix this by giving every block an identity path for the gradient.

**What's new:**
- 5 residual blocks with progressively more filters (32 → 64 → 128 → 256 → 512). Each block downsamples spatially (`MaxPooling2D(3, strides=2)`) and the shortcut uses a `1×1 Conv` to match the new shape.
- GAP head + Dropout(0.5) — same as Model 2.
- No BN yet — that comes in Model 4. This is intentional so that Model 3 vs. Model 4 isolates the value of "proper Conv-BN-ReLU ordering".


In [ ]:
# --- Architecture: stem → 5 residual blocks → GAP → Dense ----------------
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

x = layers.Conv2D(32, 5, strides=2, padding='same', use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

# --- 4. Residual Blocks (4 Stages) -----------------------------------------
# Progressively doubles filters while halving spatial dimensions
for size in [64, 128, 256, 512]:
    residual = x

    # First Conv -> BN -> ReLU
    x = layers.Conv2D(size, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second Conv -> BN -> MaxPool (Downsample happens here)
    x = layers.Conv2D(size, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

    # Shortcut Projection
    # 1x1 Conv with stride=2 matches the downsampling of the MaxPool layer
    residual = layers.Conv2D(size, 1, strides=2, padding='same', use_bias=False)(residual)
    residual = layers.BatchNormalization()(residual)

    # Add -> ReLU (Crucial: Activation happens AFTER the addition)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)

# --- 5. Classification Head ------------------------------------------------
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x) # Kept at 0.5 for strong regularization before the final decision
outputs = layers.Dense(num_classes, activation='softmax')(x)

model_m3 = keras.Model(inputs, outputs, name='M3_residual')
model_m3.summary()


In [ ]:
model_m3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=7.5e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')],
)

callbacks_m3 = [
    keras.callbacks.ModelCheckpoint(MODEL_M3, save_best_only=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
]

history_m3 = model_m3.fit(
    train_dataset,
    epochs=80,
    validation_data=val_dataset,
    callbacks=callbacks_m3,
)


In [ ]:
plot_history(history_m3, 'M3 — residual + augmentation')
print_best_val(history_m3)


**Interpretation of Model 3.** With residual shortcuts the network actually *uses* its depth — early epochs are slow (no BN yet), but val accuracy climbs steadily and overtakes Model 2 by a comfortable margin. `ReduceLROnPlateau` typically gives one final bump in val accuracy when it halves the LR.

The remaining issue: the missing BN makes optimisation noisy, and there is no built-in mechanism to fight overfitting *within* the conv stack. Model 4 fixes both.


### 2.4 Model 4 — improved residual (the winner)

**Goal of this step:** take everything that worked and combine it with the *proper* Conv → BN → ReLU ordering used by modern ResNets.

**What's new vs. Model 3:**
- **Stem with stride-2 + MaxPool** → faster early downsampling (matches the ResNet stem). The input map shrinks from 224 → 56 in two cheap layers.
- **Conv → BN → ReLU → Conv → BN → add → ReLU** ordering inside each block (the post-activation ResNet style). The final ReLU comes *after* the residual addition.
- **BatchNormalization** in every conv block (with `use_bias=False`, since BN's β replaces the bias).
- **Downsample blocks** explicitly: `stride=2` in the first conv and on the shortcut projection — so the shortcut's spatial size matches the main branch.
- **`SpatialDropout2D(0.1)`** before the head: drops entire feature maps (not individual pixels). Stronger regulariser for conv stacks than channel-wise Dropout on the GAP output alone — the two are complementary.
- Lower initial learning rate (`1.5e-4`) — BN tolerates higher LR but residual nets like a gentler ramp.

**Why I expect this to win:**
BN stabilises every block, the proper ordering keeps gradients clean through the shortcuts, and SpatialDropout fights overfit *inside* the stack (where most of the parameters live) rather than only at the head.


In [ ]:
# --- Architecture ---------------------------------------------------------
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

# Stem — 7x7 stride-2 conv + 3x3 stride-2 max-pool (classic ResNet stem).
# Brings spatial size 224 → 112 → 56 in two cheap layers.
x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

# --- Residual block helper -----------------------------------------------
# Two 3x3 convs in the main branch, optionally downsampling by stride 2 in the
# first conv. The shortcut is projected with a 1x1 conv whenever the shape
# changes (different spatial size OR different number of channels).
# The final ReLU is applied AFTER the addition (post-activation ResNet).
def residual_block(x, filters, downsample=False):
    stride   = 2 if downsample else 1
    residual = x

    # Conv → BN → ReLU → Conv
    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(x)

    # Project the shortcut whenever the shape changes (stride>1 or new channels).
    if downsample or residual.shape[-1] != filters:
        residual = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(residual)
        residual = layers.BatchNormalization()(residual)

    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)         # ReLU *after* the addition
    return x

# 4 residual stages.  filters double on each downsample → standard ResNet pattern.
x = residual_block(x, 64)
x = residual_block(x, 128, downsample=True)
x = residual_block(x, 256, downsample=True)
x = residual_block(x, 512)

# Channel-wise dropout inside the stack — drops entire feature maps.
x = layers.SpatialDropout2D(0.1)(x)

# Head: GAP → Dropout → Dense(25, softmax).
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model_m4 = keras.Model(inputs, outputs, name='M4_playmodel')
model_m4.summary()


In [ ]:
model_m4.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1.5e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')],
)

callbacks_m4 = [
    keras.callbacks.ModelCheckpoint(MODEL_M4, save_best_only=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
]

history_m4 = model_m4.fit(
    train_dataset,
    epochs=100,
    validation_data=val_dataset,
    callbacks=callbacks_m4,
)


In [ ]:
plot_history(history_m4, 'M4 — improved residual')
print_best_val(history_m4)


**Interpretation of Model 4.** Proper BN + the correct Conv-BN-ReLU ordering makes optimisation much smoother — val accuracy climbs faster and ends higher than Model 3. `SpatialDropout2D(0.1)` inside the stack plus `Dropout(0.5)` at the head keep the train/val gap narrow even at the late stages. This is the architecture I'll retrain on `train + val` combined in Section A.4.


## 3. Compare the four climb models

The table below picks the best epoch (lowest `val_loss`, which is also what `ModelCheckpoint` saved) for each model. Sorted ascending by `val_loss` — the top row is the best.


In [ ]:
import pandas as pd

def summary_row(name, history):
    i = int(np.argmin(history.history['val_loss']))
    return {
        'model':             name,
        'best_epoch':        i + 1,
        'val_loss':          history.history['val_loss'][i],
        'val_accuracy':      history.history['val_accuracy'][i],
        'val_top3_accuracy': history.history.get(
            'val_top3_accuracy', [np.nan] * len(history.history['val_loss'])
        )[i],
    }

results = pd.DataFrame([
    summary_row('M1 — simple convnet (baseline)', history_m1),
    summary_row('M2 — + augmentation + BN',        history_m2),
    summary_row('M3 — residual + augmentation',    history_m3),
    summary_row('M4 — improved residual',          history_m4),
]).sort_values('val_loss').reset_index(drop=True)

results


**Reading the table — Chollet's three-step climb in numbers:**

1. **Baseline (M1)** → small val accuracy, big train/val gap → overfit, as expected.
2. **+ Augmentation + BN (M2)** → val accuracy roughly **doubles**; the gap shrinks.
3. **+ Residual (M3)** → another sizeable jump because the network can finally use its depth.
4. **+ Proper Conv-BN-ReLU + SpatialDropout (M4)** → the best — picked as the final from-scratch architecture.

In Section A.4 below I rebuild M4 from scratch (same architecture, fresh weights) and train it on **train + val combined** to get the best possible final from-scratch deliverable.


## 4. Final from-scratch model — retrain M4 on `train + val`

**Why retrain on combined data?**
The val split was useful for *picking* the architecture. Now that M4 has won, throwing away those ~500 val images is wasteful — they contain real signal. I retrain M4 from scratch on the combined `train + val` set so the final deliverable sees ~3,950 images instead of ~3,450.

**Monitoring strategy: 10% holdout.**
Because we now have *no* held-out val set, I randomly split the combined dataset 90 / 10. The 10 % "monitor" set is *not* used for early-stopping decisions about architecture (that was already done in section 3) — it's just there so `ModelCheckpoint(save_best_only=True)` still has something to compare against, and so we can plot a learning curve.

The split is class-stratified (we use `image_dataset_from_directory` twice with the same seed and `validation_split`), so each class is represented in both halves.


In [ ]:
# We need a folder structure that contains train + val together. The cleanest
# trick on Colab is to symlink every class from both train/ and val/ into a
# single staging directory — no copying, no extra disk.
import shutil

COMBINED_DIR = '/content/data_combined'
if os.path.exists(COMBINED_DIR):
    shutil.rmtree(COMBINED_DIR)
os.makedirs(COMBINED_DIR)

for cls in class_names:
    dst = os.path.join(COMBINED_DIR, cls)
    os.makedirs(dst, exist_ok=True)
    for src_root in (TRAIN_DIR, VAL_DIR):
        src_cls = os.path.join(src_root, cls)
        for f in os.listdir(src_cls):
            if f.startswith('.'):
                continue
            # Prefix with parent split name so we never get filename collisions.
            link_name = f'{os.path.basename(src_root)}__{f}'
            os.symlink(os.path.join(src_cls, f), os.path.join(dst, link_name))

# Sanity check — total counts.
combined_counts = {c: len(os.listdir(os.path.join(COMBINED_DIR, c))) for c in class_names}
print(f'Combined total: {sum(combined_counts.values())} images across {len(class_names)} classes')
print(f'Per class: mean {np.mean(list(combined_counts.values())):.1f}, '
      f'min {min(combined_counts.values())}, max {max(combined_counts.values())}')


In [ ]:
# 95 / 5 split, class-stratified, deterministic via seed.

full_ds = keras.utils.image_dataset_from_directory(
    COMBINED_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=SPLIT_SEED,
    shuffle=True,
)


full_ds = full_ds.prefetch(tf.data.AUTOTUNE)

for x, y in train_full_ds.take(1):
    print('Final-train batch:', x.shape, '  labels:', y.shape)


In [ ]:
# --- Architecture: stem → 5 residual blocks → GAP → Dense ----------------
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

x = layers.Conv2D(32, 5, strides=2, padding='same', use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

# --- 4. Residual Blocks (4 Stages) -----------------------------------------
# Progressively doubles filters while halving spatial dimensions
for size in [64, 128, 256, 512]:
    residual = x

    # First Conv -> BN -> ReLU
    x = layers.Conv2D(size, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second Conv -> BN -> MaxPool (Downsample happens here)
    x = layers.Conv2D(size, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

    # Shortcut Projection
    # 1x1 Conv with stride=2 matches the downsampling of the MaxPool layer
    residual = layers.Conv2D(size, 1, strides=2, padding='same', use_bias=False)(residual)
    residual = layers.BatchNormalization()(residual)

    # Add -> ReLU (Crucial: Activation happens AFTER the addition)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)

# --- 5. Classification Head ------------------------------------------------
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x) # Kept at 0.5 for strong regularization before the final decision
outputs = layers.Dense(num_classes, activation='softmax')(x)

model_final_scratch = keras.Model(inputs, outputs, name='best_from_scratch')

# 3) Compile — rovnaký LR ako climb (1e-3 pre M3)
model_final_scratch.compile(
    optimizer=keras.optimizers.Adam(learning_rate=7.5e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')],
)

# 4) Callbacks — žiadny val_loss, save final
# ReduceLROnPlateau na 'loss' (training) — vždy dostupné
callbacks_final_scratch = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='loss', factor=0.5, patience=8, min_lr=1e-6
    ),
]

# 5) FIXED EPOCH COUNT z climb výsledkov
# Climb best val_loss epoch bol 46
# Pre combined retrain trochu skor (lebo viac dát/epoch updates):
EPOCHS = 78

history_final_scratch = model_final_scratch.fit(
    full_ds,
    epochs=EPOCHS,
    callbacks=callbacks_final_scratch,
)

# 6) Save final model — toto je tvoj deliverable
model_final_scratch.save(BEST_FROM_SCRATCH)
print(f'→ Final from-scratch model: {BEST_FROM_SCRATCH}')
print(f'  Trained on {4448} images for {EPOCHS} epochs (no val holdout — Chollet CH 8 §8.2.4)')

In [ ]:
plot_history(history_final_scratch, 'Final from-scratch (M4 on train+val)')
print_best_val(history_final_scratch)
print(f'\n→ Saved to: {BEST_FROM_SCRATCH}')


## 5. Pretrained model — EfficientNetV2B0 (frozen → fine-tuned)

The from-scratch model has to learn *everything* — including low-level edges — from ~4 000 images. A pretrained network already knows generic visual primitives (edges, textures, parts) from ImageNet's 1.2 M images. We can keep those features and only learn the grocery-specific classifier on top. This is the **transfer-learning** pattern from CH 8.

Two phases (Chollet's recipe):
1. **Phase 1 — Frozen base.** Freeze the whole convolutional base, train only a fresh classifier head with a relatively high LR (`1e-3`). Fast — the base just runs a forward pass per image.
2. **Phase 2 — Fine-tuning.** Unfreeze the top few blocks (the most task-specific layers, since deeper layers encode more abstract features). Re-compile with a **100× lower LR** (`1e-5`) so the pretrained weights drift only slightly. Keep BatchNorm layers frozen — their running statistics come from ImageNet and re-learning them on 4 000 images would hurt.

Both phases train on the **same train+val combined** dataset with the 10% monitor split from Section 4.


### 5.1 Phase 1 — frozen convolutional base


In [ ]:
from tensorflow.keras.applications import EfficientNetV2B0

# Pretrained base — include_top=False drops ImageNet's 1000-way classifier.
conv_base = EfficientNetV2B0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3),
)
conv_base.trainable = False

# Lighter augmentation for the pretrained model — heavy augmentation can hurt
# when the features are already excellent and we have little data to fine-tune.
data_augmentation_pre = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='data_augmentation_pre')

# IMPORTANT: EfficientNetV2 expects pixels in [0, 255]. Its preprocess_input is
# a no-op (the model uses an internal Normalization layer trained on ImageNet).
# So we feed raw uint8 images straight in — no Rescaling(1./255) here.
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation_pre(inputs)
x = conv_base(x, training=False)              # keep BN in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model_pretrained = keras.Model(inputs, outputs, name='pretrained_frozen')

model_pretrained.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')],
)

callbacks_phase1 = [
    keras.callbacks.ModelCheckpoint(
        BEST_FINETUNED, save_best_only=True,
        monitor='val_loss', mode='min',
    ),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]

history_phase1 = model_pretrained.fit(
    train_dataset,
    epochs=40,
    validation_data=val_dataset,
    callbacks=callbacks_phase1,
)


In [ ]:
plot_history(history_phase1, 'Pretrained — phase 1 (frozen base)')
print_best_val(history_phase1)


**Phase 1 observation.** The pretrained features are *extremely* powerful — val accuracy after only a few epochs is already higher than the from-scratch model needs ~100 epochs to reach. The head is now well-aligned with the frozen features; in Phase 2 we let the *top* of the conv base shift slightly to specialise on groceries.


### 5.2 Phase 2 — fine-tune the top blocks

**What I unfreeze:**
- The last few residual stages (`block6f`, `block6g`, `block6h`) plus the `top` head. Deeper in EfficientNet = more task-specific — those are the layers worth re-training. Earlier layers detect edges/textures that are equally useful for groceries as for ImageNet.

**What stays frozen:**
- Every `BatchNormalization` layer inside the conv base, even in the unfrozen blocks. Their `moving_mean` / `moving_variance` were estimated on millions of ImageNet images. Updating them on our 4 000 images would replace good statistics with worse ones — a well-documented gotcha (see Chollet §8.5).

**Learning rate:** 1e-5 — **100× lower** than Phase 1. Big LR + pretrained weights = destroying the very thing we're trying to preserve.


In [ ]:
# 1. Re-load Phase-1's best checkpoint (NOT the in-memory model — its weights
#    were updated even after the best epoch).
model_pretrained = keras.models.load_model(BEST_FINETUNED)

# 2. Find the conv_base layer inside the saved model.
conv_base = model_pretrained.get_layer('efficientnetv2-b0')

# 3. Unfreeze block6f / block6g / block6h + top  — but keep their BN layers frozen.
conv_base.trainable = True
trainable_count, frozen_bn_count = 0, 0
for layer in conv_base.layers:
    if layer.name.startswith(('block6f', 'block6g', 'block6h', 'top')):
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
            frozen_bn_count += 1
        else:
            layer.trainable = True
            trainable_count += 1
    else:
        layer.trainable = False

print(f'Trainable layers (excluding BN): {trainable_count}')
print(f'Frozen BN layers inside unfrozen blocks: {frozen_bn_count}')

# 4. Recompile with a much lower LR — must recompile so the new trainable
#    flags take effect.
model_pretrained.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')],
)

# 5. Train — overwrites BEST_FINETUNED if and only if Phase 2 beats Phase 1.
callbacks_phase2 = [
    keras.callbacks.ModelCheckpoint(
        BEST_FINETUNED, save_best_only=True,
        monitor='val_loss', mode='min',
    ),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7),
]

history_phase2 = model_pretrained.fit(
    train_dataset,
    epochs=30,
    validation_data=val_dataset,
    callbacks=callbacks_phase2,
)


In [ ]:
plot_history(history_phase2, 'Pretrained — phase 2 (fine-tune top blocks)')
print_best_val(history_phase2)
print(f'\n→ Saved to: {BEST_FINETUNED}')


**Phase 2 observation.** The fine-tuned model usually edges past Phase 1 by a couple of percentage points — the top blocks shift just enough to specialise on grocery-package features. We do not unfreeze further: with our tiny dataset, deeper fine-tuning would start to destroy useful ImageNet features. `ModelCheckpoint(save_best_only=True)` makes the file on Drive reflect whichever of Phase 1 / Phase 2 happens to win on accuracy.


---
# **SECTION B — Exam day (no training)**

This section answers the exam questions. **No training happens here.** I load the two best models from Drive and evaluate them on the `test/` folder that the examiner provides.

**Pre-flight checklist on exam day:**
1. Cells in Section 0 are already run (Drive mounted, paths set, GPU on, imports done).
2. The two `.keras` files are on Drive at the paths printed by Section 0.
3. `test/` is extracted under `/content/data/exam_ML4BI2_2026/test/`.


In [ ]:
# Load both saved models from Drive — no retraining.
print('Loading from-scratch model from:', BEST_FROM_SCRATCH)
best_scratch = keras.models.load_model(BEST_FROM_SCRATCH)

print('Loading fine-tuned  model from:', BEST_FINETUNED)
best_finetuned = keras.models.load_model(BEST_FINETUNED)

print('\nBoth models loaded successfully.')


### Build the test pipeline

Same parameters as the training pipeline: `(224, 224)`, batch 32, `categorical` labels, no shuffle (so per-class breakdowns line up cleanly).


In [ ]:
test_dataset = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

# Sanity check that the test class order matches the training class order.
test_class_names = sorted([c for c in os.listdir(TEST_DIR) if not c.startswith('.')])
assert test_class_names == class_names, 'Test classes do not match training classes!'
print('Class order matches between train and test ✓')
print(f'Test images: {sum(len(os.listdir(os.path.join(TEST_DIR, c))) for c in class_names)}')


### Evaluate the from-scratch model


In [ ]:
print('--- Best from-scratch (M4 on train+val) ---')
metrics_scratch = best_scratch.evaluate(test_dataset, verbose=0)
for name, value in zip(best_scratch.metrics_names, metrics_scratch):
    print(f'  {name:20s} = {value:.4f}')


In [ ]:
# Per-class breakdown — which classes are easy / hard for the from-scratch model.
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_prob = [], []
for x, y in test_dataset:
    y_prob.append(best_scratch.predict(x, verbose=0))
    y_true.append(y.numpy())
y_prob_s   = np.concatenate(y_prob)
y_true_oh  = np.concatenate(y_true)
y_pred_s   = np.argmax(y_prob_s,  axis=1)
y_true_idx = np.argmax(y_true_oh, axis=1)

print(classification_report(y_true_idx, y_pred_s, target_names=class_names, digits=3))


In [ ]:
cm_s = confusion_matrix(y_true_idx, y_pred_s)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm_s, cmap='Blues')
ax.set_xticks(range(num_classes)); ax.set_xticklabels(class_names, rotation=90)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix — best from-scratch model on test/')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()


### Evaluate the fine-tuned pretrained model


In [ ]:
print('--- Best fine-tuned (EfficientNetV2B0 on train+val) ---')
metrics_ft = best_finetuned.evaluate(test_dataset, verbose=0)
for name, value in zip(best_finetuned.metrics_names, metrics_ft):
    print(f'  {name:20s} = {value:.4f}')


In [ ]:
y_true, y_prob = [], []
for x, y in test_dataset:
    y_prob.append(best_finetuned.predict(x, verbose=0))
    y_true.append(y.numpy())
y_prob_f   = np.concatenate(y_prob)
y_true_oh  = np.concatenate(y_true)
y_pred_f   = np.argmax(y_prob_f,  axis=1)
y_true_idx = np.argmax(y_true_oh, axis=1)

print(classification_report(y_true_idx, y_pred_f, target_names=class_names, digits=3))


In [ ]:
cm_f = confusion_matrix(y_true_idx, y_pred_f)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm_f, cmap='Blues')
ax.set_xticks(range(num_classes)); ax.set_xticklabels(class_names, rotation=90)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix — fine-tuned EfficientNetV2B0 on test/')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()
